**Now we move from Silver → Gold.**

**The Silver table contains cleaned, deduplicated orders. The Gold layer converts that data into business-ready sales KPIs.**

```text
SILVER
   │
   │ business aggregation
   ↓
GOLD_REVENUE
   │
   ├── Revenue
   ├── Orders
   └── Units
```

### Create Gold schema

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS retail_lakehouse.gold;

### Gold revenue table

In [0]:
%sql
CREATE OR REPLACE TABLE retail_lakehouse.gold.gold_revenue
USING DELTA
AS

SELECT
    order_date,

    SUM(quantity * unit_price) AS total_revenue,

    SUM(quantity) AS total_units,

    COUNT(DISTINCT order_id) AS total_orders,

    ROUND(
        SUM(quantity * unit_price)
        / COUNT(DISTINCT order_id),
        2
    ) AS average_order_value

FROM retail_lakehouse.silver.orders

GROUP BY order_date

ORDER BY order_date;

num_affected_rows,num_inserted_rows


### Validate the Gold table

In [0]:
%sql
SELECT *
FROM retail_lakehouse.gold.gold_revenue
ORDER BY order_date;

order_date,total_revenue,total_units,total_orders,average_order_value
2026-08-01,588380.01,2315,520,1131.50
2026-08-02,566560.87,2242,524,1081.22
2026-08-03,593796.18,2316,517,1148.54
2026-08-04,600878.81,2395,516,1164.49
2026-08-05,542053.31,2184,492,1101.73
2026-08-06,583228.43,2324,494,1180.62
2026-08-07,607010.42,2348,518,1171.83
2026-08-08,563441.86,2209,499,1129.14
2026-08-09,585798.47,2270,512,1144.14
2026-08-10,620713.79,2481,549,1130.63


### Some business validation
We should make sure we don't have negative revenue.

In [0]:
%sql
SELECT
    COUNT(*) AS invalid_rows
FROM retail_lakehouse.gold.gold_revenue
WHERE total_revenue < 0;

invalid_rows
0


### Calculate overall KPIs

These queries will eventually be useful for the dashboard.

**Total revenue**

In [0]:
%sql
SELECT
    ROUND(SUM(total_revenue), 2) AS total_revenue
FROM retail_lakehouse.gold.gold_revenue;

total_revenue
15749780.82


**Total orders**

In [0]:
%sql
SELECT
    SUM(total_orders) AS total_orders
FROM retail_lakehouse.gold.gold_revenue;

total_orders
13713


**Total units**

In [0]:
%sql
SELECT
    SUM(total_units) AS total_units
FROM retail_lakehouse.gold.gold_revenue;

total_units
62070
